# Conservation Threshold Sweep & Max-Pooling Residue Analysis

**Goal:** Understand why max pooling outperforms mean pooling for non-conserved enzyme residues.

- **Analysis A:** Sweep 8 conservation thresholds x 2 pooling strategies x 2 models x 10 seeds = 320 experiments
- **Analysis B:** For the best threshold/max-pool combo, identify which residues "win" the max pooling across 1024 dimensions

## Section 1: Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from Bio import SeqIO

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

SWEEP_DIR = OUTPUT_DIR / "model_outputs" / "threshold_sweep"
SWEEP_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
POOLING_METHODS = ['mean', 'max']

print(f"Thresholds: {THRESHOLDS}")
print(f"Pooling: {POOLING_METHODS}")
print(f"Seeds: {SEEDS}")
print(f"Total experiments: {len(THRESHOLDS)} x {len(POOLING_METHODS)} x 2 models x {N_SPLITS} seeds = {len(THRESHOLDS)*len(POOLING_METHODS)*2*N_SPLITS}")

## Section 2: Load Data

In [ ]:
%%time
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]  # Strip species prefix
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")
sample_key = list(per_residue_embeddings.keys())[0]
print(f"  Example: {sample_key} shape {per_residue_embeddings[sample_key].shape}")

In [ ]:
# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
print(f"Conservation scores: {len(df_cons)} alignment positions")
print(df_cons.head())

# Keep only core columns
df_core = df_cons[['alignment_position', 'conservation_score']].copy()
print(f"\nConservation stats:")
print(df_core['conservation_score'].describe())

In [ ]:
# --- MSA alignment mapping ---
alignment_to_seq = {}
fasta_path = OUTPUT_DIR / "bsh_aligned.fasta"
for record in SeqIO.parse(fasta_path, "fasta"):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id  # Strip species prefix
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    for aln_pos, char in enumerate(seq):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
print(f"Alignment mappings: {len(alignment_to_seq)} enzymes")

In [ ]:
# --- Activity data ---
df_act = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
print(f"Activity data: {len(df_act)} pairs")
print(f"Columns: {list(df_act.columns)}")

# Aggregate to pair-level (approach 2: any replicate active -> active)
df_agg = df_act.groupby(['Enzyme', 'Amine']).agg({'active_approach2': 'max'}).reset_index()
df_agg.rename(columns={'active_approach2': 'active'}, inplace=True)
df_agg['active'] = df_agg['active'].astype(int)
print(f"Aggregated: {len(df_agg)} unique enzyme-amine pairs")
print(f"Active: {df_agg['active'].sum()} ({df_agg['active'].mean():.1%})")

In [ ]:
# --- Amine SMILES and features (physchem_onehot) ---
smiles_df = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")
print(f"SMILES file: {len(smiles_df)} compounds")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

smiles_dict = {}
for _, row in smiles_df.iterrows():
    raw_name = str(row.iloc[0]).strip()
    smiles = str(row.iloc[1]).strip()
    name = name_map.get(raw_name, raw_name.lower())
    smiles_dict[name] = smiles

all_amines_sorted = sorted(df_agg['Amine'].unique())
print(f"Activity amines: {len(all_amines_sorted)}")

# Physicochemical descriptors
def compute_physicochemical(mol):
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol),
        rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)

# Build physchem + one-hot amine features
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
amine_features = {}

for amine in all_amines_sorted:
    onehot = np.zeros(len(all_amines_sorted), dtype=np.float32)
    onehot[amine_to_idx[amine]] = 1.0
    if amine in smiles_dict:
        mol = Chem.MolFromSmiles(smiles_dict[amine])
        if mol is not None:
            physchem = compute_physicochemical(mol)
        else:
            physchem = np.zeros(15, dtype=np.float32)
    else:
        physchem = np.zeros(15, dtype=np.float32)
    amine_features[amine] = np.concatenate([physchem, onehot])

sample = list(amine_features.values())[0]
print(f"Amine feature dim: {len(sample)} (15 physchem + {len(all_amines_sorted)} one-hot)")

In [ ]:
# --- Find overlap of enzymes across all data sources ---
enzymes_h5 = set(per_residue_embeddings.keys())
enzymes_aln = set(alignment_to_seq.keys())
enzymes_act = set(df_agg['Enzyme'].unique())

overlap = enzymes_h5 & enzymes_aln & enzymes_act
print(f"Enzyme overlap: {len(overlap)} enzymes")
print(f"  H5: {len(enzymes_h5)}, Alignment: {len(enzymes_aln)}, Activity: {len(enzymes_act)}")

# Debug: if overlap is small, show sample IDs from each source
if len(overlap) < 50:
    print(f"\n  Sample H5 IDs: {sorted(enzymes_h5)[:5]}")
    print(f"  Sample Alignment IDs: {sorted(enzymes_aln)[:5]}")
    print(f"  Sample Activity IDs: {sorted(enzymes_act)[:5]}")
    # Check pairwise overlaps to find the mismatch
    print(f"\n  H5 & Alignment: {len(enzymes_h5 & enzymes_aln)}")
    print(f"  H5 & Activity: {len(enzymes_h5 & enzymes_act)}")
    print(f"  Alignment & Activity: {len(enzymes_aln & enzymes_act)}")

## Section 3: Residue Tracking Per Threshold

In [ ]:
def get_nonconserved_embedding(enzyme_id, conservation_threshold, pooling='mean'):
    """Extract and pool per-residue embeddings at non-conserved positions."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    elif pooling == 'mean_max':
        return np.concatenate([selected.mean(axis=0), selected.max(axis=0)])
    return selected.mean(axis=0)

In [ ]:
# Track which alignment positions are non-conserved at each threshold,
# and how many sequence positions each enzyme retains

residue_count_rows = []
residue_position_rows = []

for thresh in THRESHOLDS:
    # Alignment positions below threshold
    variable_aln = df_core[
        df_core['conservation_score'] < thresh
    ]['alignment_position'].values
    n_aln_positions = len(variable_aln)

    # Per-enzyme: count of sequence positions selected
    per_enzyme_counts = []
    for eid in sorted(overlap):
        aln_map = alignment_to_seq[eid]
        seq_positions = []
        for aln_pos in variable_aln:
            if aln_pos in aln_map:
                seq_pos = aln_map[aln_pos]
                if seq_pos < len(per_residue_embeddings[eid]):
                    seq_positions.append(seq_pos)
        per_enzyme_counts.append(len(seq_positions))

    residue_count_rows.append({
        'threshold': thresh,
        'n_alignment_positions': n_aln_positions,
        'per_enzyme_min': int(np.min(per_enzyme_counts)),
        'per_enzyme_mean': float(np.mean(per_enzyme_counts)),
        'per_enzyme_max': int(np.max(per_enzyme_counts)),
        'per_enzyme_std': float(np.std(per_enzyme_counts)),
    })

    # Store alignment positions for this threshold
    for aln_pos in variable_aln:
        cons_score = df_core.loc[
            df_core['alignment_position'] == aln_pos, 'conservation_score'
        ].values[0]
        residue_position_rows.append({
            'threshold': thresh,
            'alignment_position': aln_pos,
            'conservation_score': cons_score,
        })

df_residue_counts = pd.DataFrame(residue_count_rows)
df_residue_positions = pd.DataFrame(residue_position_rows)

# Save
df_residue_counts.to_csv(SWEEP_DIR / "residue_counts_per_threshold.csv", index=False)
df_residue_positions.to_csv(SWEEP_DIR / "residue_positions_per_threshold.csv", index=False)

print("Residue counts per threshold:")
print(df_residue_counts.to_string(index=False))

In [ ]:
# Figure: residue count vs threshold
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: alignment positions
ax = axes[0]
ax.plot(df_residue_counts['threshold'], df_residue_counts['n_alignment_positions'],
        'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Conservation Threshold', fontsize=12)
ax.set_ylabel('Non-Conserved Alignment Positions', fontsize=12)
ax.set_title('Alignment Positions Below Threshold', fontsize=13)
ax.grid(True, alpha=0.3)

# Right: per-enzyme sequence positions (mean +/- std)
ax = axes[1]
ax.fill_between(
    df_residue_counts['threshold'],
    df_residue_counts['per_enzyme_min'],
    df_residue_counts['per_enzyme_max'],
    alpha=0.15, color='steelblue', label='min-max range'
)
ax.fill_between(
    df_residue_counts['threshold'],
    df_residue_counts['per_enzyme_mean'] - df_residue_counts['per_enzyme_std'],
    df_residue_counts['per_enzyme_mean'] + df_residue_counts['per_enzyme_std'],
    alpha=0.3, color='steelblue', label='mean +/- std'
)
ax.plot(df_residue_counts['threshold'], df_residue_counts['per_enzyme_mean'],
        'o-', color='steelblue', linewidth=2, markersize=8, label='mean')
ax.set_xlabel('Conservation Threshold', fontsize=12)
ax.set_ylabel('Sequence Positions Per Enzyme', fontsize=12)
ax.set_title('Per-Enzyme Residues Selected', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "residue_count_vs_threshold.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: residue_count_vs_threshold.png")

## Section 4: Build Feature Matrices

In [ ]:
%%time
# Build enzyme embeddings for each (threshold, pooling) combo
feature_matrices = {}

for thresh in THRESHOLDS:
    for pooling in POOLING_METHODS:
        key = (thresh, pooling)

        # Compute enzyme embeddings
        enz_dict = {}
        for eid in overlap:
            emb = get_nonconserved_embedding(eid, thresh, pooling=pooling)
            if emb is not None:
                enz_dict[eid] = emb

        # Build feature matrix
        X_list, y_list = [], []
        enzymes_list, amines_list = [], []
        for _, row in df_agg.iterrows():
            enzyme, amine = row['Enzyme'], row['Amine']
            if enzyme not in enz_dict or amine not in amine_features:
                continue
            features = np.concatenate([enz_dict[enzyme], amine_features[amine]])
            X_list.append(features)
            y_list.append(int(row['active']))
            enzymes_list.append(enzyme)
            amines_list.append(amine)

        X = np.array(X_list, dtype=np.float32)
        y = np.array(y_list, dtype=np.int32)
        feature_matrices[key] = (X, y, enzymes_list, amines_list)

        print(f"  thresh={thresh}, pool={pooling}: {X.shape[0]} samples, "
              f"{X.shape[1]} features, {len(enz_dict)} enzymes")

print(f"\nTotal feature matrices: {len(feature_matrices)}")

## Section 5: Train Models (320 experiments)

In [ ]:
def enzyme_holdout_split_seed(X, y, enzymes, amines, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with a specific random seed."""
    enzymes_arr = np.array(enzymes)
    amines_arr = np.array(amines)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins
    )
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins
    )
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
        'test_enzymes_arr': enzymes_arr[test_mask],
        'test_amines_arr': amines_arr[test_mask],
    }


def train_xgb(split_data):
    """Train regularized XGBoost. Returns metrics dict and model."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']

    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)

    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=n_neg / n_pos,
        reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5,
        random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]

    return {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
    }, model


def train_mlp(split_data):
    """Train MLP with StandardScaler. Returns metrics dict and model."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    # Combine train+val for MLP training (use internal early stopping split)
    X_trainval_s = np.vstack([X_train_s, X_val_s])
    y_trainval = np.concatenate([y_train, y_val])
    val_frac = len(y_val) / len(y_trainval)

    model = MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation='relu',
        max_iter=300,
        early_stopping=True,
        validation_fraction=val_frac,
        n_iter_no_change=20,
        random_state=42,
        batch_size=64,
        learning_rate='adaptive',
        learning_rate_init=0.001,
        alpha=0.01,
    )
    model.fit(X_trainval_s, y_trainval)

    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    y_proba_train = model.predict_proba(X_train_s)[:, 1]
    y_proba_val = model.predict_proba(X_val_s)[:, 1]

    return {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
    }, model

In [ ]:
%%time
# Main training loop: 320 experiments
all_results = []
all_models_xgb = {}  # (thresh, pooling, seed) -> model

total = len(THRESHOLDS) * len(POOLING_METHODS) * 2 * N_SPLITS
done = 0

for thresh in THRESHOLDS:
    for pooling in POOLING_METHODS:
        key = (thresh, pooling)
        X, y, enzymes, amines = feature_matrices[key]

        for i, seed in enumerate(SEEDS):
            split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)

            # XGBoost
            metrics_xgb, model_xgb = train_xgb(split)
            metrics_xgb.update({
                'threshold': thresh, 'pooling': pooling,
                'model': 'XGBoost', 'seed': seed, 'split_idx': i,
            })
            all_results.append(metrics_xgb)
            all_models_xgb[(thresh, pooling, seed)] = model_xgb

            # MLP
            metrics_mlp, _ = train_mlp(split)
            metrics_mlp.update({
                'threshold': thresh, 'pooling': pooling,
                'model': 'MLP', 'seed': seed, 'split_idx': i,
            })
            all_results.append(metrics_mlp)

            done += 2

        print(f"  thresh={thresh}, pool={pooling}: done ({done}/{total})")

df_results = pd.DataFrame(all_results)
df_results.to_csv(SWEEP_DIR / "threshold_sweep_results.csv", index=False)
print(f"\nTotal results: {len(df_results)} rows")
print("Saved: threshold_sweep_results.csv")

## Section 6: Visualize Threshold Sweep

In [ ]:
# Build summary table
summary_rows = []
for thresh in THRESHOLDS:
    for pooling in POOLING_METHODS:
        for model_name in ['XGBoost', 'MLP']:
            mask = (
                (df_results['threshold'] == thresh) &
                (df_results['pooling'] == pooling) &
                (df_results['model'] == model_name)
            )
            sub = df_results[mask]
            n_residues = df_residue_counts.loc[
                df_residue_counts['threshold'] == thresh, 'per_enzyme_mean'
            ].values[0]
            summary_rows.append({
                'threshold': thresh,
                'pooling': pooling,
                'model': model_name,
                'n_residues_mean': n_residues,
                'roc_auc_mean': sub['roc_auc'].mean(),
                'roc_auc_std': sub['roc_auc'].std(),
                'pr_auc_mean': sub['pr_auc'].mean(),
                'pr_auc_std': sub['pr_auc'].std(),
                'f1_mean': sub['f1'].mean(),
                'f1_std': sub['f1'].std(),
                'logloss_gap_mean': sub['logloss_gap'].mean(),
                'logloss_gap_std': sub['logloss_gap'].std(),
            })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(SWEEP_DIR / "threshold_sweep_summary.csv", index=False)
print("Summary:")
print(df_summary.to_string(index=False))

In [ ]:
# Figure 1: ROC-AUC and PR-AUC vs threshold
# Lines for mean/max, panels for XGBoost/MLP
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'mean': 'steelblue', 'max': 'coral'}

for col_idx, metric in enumerate(['roc_auc', 'pr_auc']):
    for row_idx, model_name in enumerate(['XGBoost', 'MLP']):
        ax = axes[row_idx, col_idx]
        for pooling in POOLING_METHODS:
            mask = (df_summary['pooling'] == pooling) & (df_summary['model'] == model_name)
            sub = df_summary[mask].sort_values('threshold')
            ax.errorbar(
                sub['threshold'], sub[f'{metric}_mean'],
                yerr=sub[f'{metric}_std'],
                marker='o', linewidth=2, markersize=7, capsize=4,
                color=colors[pooling], label=f'{pooling} pool'
            )
        ax.set_xlabel('Conservation Threshold', fontsize=11)
        ylabel = 'ROC-AUC' if metric == 'roc_auc' else 'PR-AUC'
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'{model_name} -- {ylabel}', fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "threshold_sweep_lines.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: threshold_sweep_lines.png")

In [ ]:
# Figure 2: Max vs Mean performance gap at each threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, model_name in enumerate(['XGBoost', 'MLP']):
    ax = axes[ax_idx]
    for metric, color, ls in [('roc_auc', 'steelblue', '-'), ('pr_auc', 'coral', '--')]:
        gaps = []
        for thresh in THRESHOLDS:
            max_val = df_summary[
                (df_summary['threshold'] == thresh) &
                (df_summary['pooling'] == 'max') &
                (df_summary['model'] == model_name)
            ][f'{metric}_mean'].values[0]
            mean_val = df_summary[
                (df_summary['threshold'] == thresh) &
                (df_summary['pooling'] == 'mean') &
                (df_summary['model'] == model_name)
            ][f'{metric}_mean'].values[0]
            gaps.append(max_val - mean_val)
        label = 'ROC-AUC' if metric == 'roc_auc' else 'PR-AUC'
        ax.plot(THRESHOLDS, gaps, f'o{ls}', color=color, linewidth=2,
                markersize=7, label=f'{label} gap')

    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Conservation Threshold', fontsize=11)
    ax.set_ylabel('Max - Mean (performance gap)', fontsize=11)
    ax.set_title(f'{model_name}: Max-Pool Advantage', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "threshold_sweep_gap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: threshold_sweep_gap.png")

In [ ]:
# Figure 3: Max-pool advantage vs number of residues
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, model_name in enumerate(['XGBoost', 'MLP']):
    ax = axes[ax_idx]
    n_res_vals = []
    roc_gaps = []
    pr_gaps = []

    for thresh in THRESHOLDS:
        n_res = df_residue_counts.loc[
            df_residue_counts['threshold'] == thresh, 'per_enzyme_mean'
        ].values[0]
        n_res_vals.append(n_res)

        for metric, gap_list in [('roc_auc', roc_gaps), ('pr_auc', pr_gaps)]:
            max_v = df_summary[
                (df_summary['threshold'] == thresh) &
                (df_summary['pooling'] == 'max') &
                (df_summary['model'] == model_name)
            ][f'{metric}_mean'].values[0]
            mean_v = df_summary[
                (df_summary['threshold'] == thresh) &
                (df_summary['pooling'] == 'mean') &
                (df_summary['model'] == model_name)
            ][f'{metric}_mean'].values[0]
            gap_list.append(max_v - mean_v)

    ax.scatter(n_res_vals, roc_gaps, s=80, color='steelblue', zorder=3, label='ROC-AUC gap')
    ax.scatter(n_res_vals, pr_gaps, s=80, color='coral', marker='s', zorder=3, label='PR-AUC gap')

    # Add threshold labels
    for j, thresh in enumerate(THRESHOLDS):
        ax.annotate(f'{thresh}', (n_res_vals[j], roc_gaps[j]),
                    textcoords='offset points', xytext=(5, 5), fontsize=8)

    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Mean Residues Per Enzyme', fontsize=11)
    ax.set_ylabel('Max - Mean Gap', fontsize=11)
    ax.set_title(f'{model_name}: Pooling Advantage vs Residue Count', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "pooling_advantage_vs_n_residues.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pooling_advantage_vs_n_residues.png")

In [ ]:
# Summary: best threshold per pooling method per model
print("=" * 80)
print("Best configurations:")
print("=" * 80)
for model_name in ['XGBoost', 'MLP']:
    print(f"\n--- {model_name} ---")
    for metric in ['roc_auc_mean', 'pr_auc_mean']:
        for pooling in POOLING_METHODS:
            mask = (df_summary['pooling'] == pooling) & (df_summary['model'] == model_name)
            sub = df_summary[mask]
            best_idx = sub[metric].idxmax()
            best = sub.loc[best_idx]
            metric_label = metric.replace('_mean', '').upper().replace('_', '-')
            print(f"  Best {metric_label} ({pooling}): "
                  f"threshold={best['threshold']:.2f}, "
                  f"{metric_label}={best[metric]:.4f} +/- {best[metric.replace('mean','std')]:.4f}")

## Section 7: Max-Contributing Residue Analysis

In [ ]:
# Find the best (threshold, max) combo based on XGBoost ROC-AUC
mask = (df_summary['pooling'] == 'max') & (df_summary['model'] == 'XGBoost')
best_row = df_summary[mask].sort_values('roc_auc_mean', ascending=False).iloc[0]
best_thresh = best_row['threshold']
print(f"Best (threshold, max) combo: threshold={best_thresh}")
print(f"  ROC-AUC: {best_row['roc_auc_mean']:.4f} +/- {best_row['roc_auc_std']:.4f}")
print(f"  PR-AUC: {best_row['pr_auc_mean']:.4f} +/- {best_row['pr_auc_std']:.4f}")

In [ ]:
# For each enzyme, find which residue positions contribute the max value
# across each of the 1024 embedding dimensions

variable_aln_positions = df_core[
    df_core['conservation_score'] < best_thresh
]['alignment_position'].values

# Track: for each (enzyme, dim), which alignment position provides the max
max_position_counts = {}  # alignment_position -> count
n_enzymes_analyzed = 0

for eid in sorted(overlap):
    if eid not in per_residue_embeddings or eid not in alignment_to_seq:
        continue

    embed = per_residue_embeddings[eid]
    aln_map = alignment_to_seq[eid]

    # Get non-conserved positions (alignment -> sequence)
    aln_to_seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                aln_to_seq_positions.append((aln_pos, seq_pos))

    if len(aln_to_seq_positions) == 0:
        continue

    aln_positions = [x[0] for x in aln_to_seq_positions]
    seq_positions = [x[1] for x in aln_to_seq_positions]

    selected = embed[seq_positions]  # shape: (n_residues, 1024)

    # For each of 1024 dims, find which residue provides the max
    argmax_per_dim = np.argmax(selected, axis=0)  # shape: (1024,)

    # Map back to alignment positions
    for dim_idx in range(1024):
        winner_local_idx = argmax_per_dim[dim_idx]
        winner_aln_pos = aln_positions[winner_local_idx]
        max_position_counts[winner_aln_pos] = max_position_counts.get(winner_aln_pos, 0) + 1

    n_enzymes_analyzed += 1

print(f"Analyzed {n_enzymes_analyzed} enzymes")
print(f"Unique alignment positions contributing max values: {len(max_position_counts)}")
print(f"Total max contributions: {sum(max_position_counts.values())} "
      f"(expected: {n_enzymes_analyzed} x 1024 = {n_enzymes_analyzed * 1024})")

In [ ]:
# Build importance DataFrame and cross-reference with conservation scores
importance_rows = []
for aln_pos, count in max_position_counts.items():
    cons_score = df_core.loc[
        df_core['alignment_position'] == aln_pos, 'conservation_score'
    ].values[0]
    importance_rows.append({
        'alignment_position': aln_pos,
        'max_frequency': count,
        'conservation_score': cons_score,
        'frequency_pct': count / (n_enzymes_analyzed * 1024) * 100,
    })

df_importance = pd.DataFrame(importance_rows).sort_values('max_frequency', ascending=False)
df_importance.to_csv(SWEEP_DIR / "max_residue_importance.csv", index=False)

print(f"Top 20 max-contributing alignment positions:")
print(df_importance.head(20).to_string(index=False))

print(f"\n--- Hot spots (top 10%) ---")
top_10pct = df_importance.head(max(1, len(df_importance) // 10))
print(f"Top {len(top_10pct)} positions account for "
      f"{top_10pct['frequency_pct'].sum():.1f}% of all max contributions")
print(f"Mean conservation score of hot spots: {top_10pct['conservation_score'].mean():.3f}")
print(f"Mean conservation score overall: {df_importance['conservation_score'].mean():.3f}")

In [ ]:
# Figure: Bar chart of alignment positions by frequency, colored by conservation score
top_n = min(50, len(df_importance))
df_top = df_importance.head(top_n)

fig, ax = plt.subplots(figsize=(16, 6))

# Color by conservation score
cmap = plt.cm.RdYlGn_r  # Red = low conservation (variable), Green = high
norm = plt.Normalize(vmin=0, vmax=best_thresh)
bar_colors = [cmap(norm(c)) for c in df_top['conservation_score']]

bars = ax.bar(
    range(top_n),
    df_top['max_frequency'],
    color=bar_colors, edgecolor='gray', linewidth=0.3
)

ax.set_xticks(range(top_n))
ax.set_xticklabels(df_top['alignment_position'].values, rotation=90, fontsize=7)
ax.set_xlabel('Alignment Position', fontsize=12)
ax.set_ylabel('Max-Contribution Frequency', fontsize=12)
ax.set_title(
    f'Top {top_n} Max-Contributing Residue Positions '
    f'(threshold={best_thresh}, {n_enzymes_analyzed} enzymes x 1024 dims)',
    fontsize=13
)
ax.grid(True, alpha=0.2, axis='y')

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Conservation Score', fontsize=11)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "max_contributing_residues.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: max_contributing_residues.png")

In [ ]:
# Additional analysis: per-enzyme concentration of max contributions
# How many positions account for 50% / 90% of max contributions?

df_sorted = df_importance.sort_values('max_frequency', ascending=False)
cumsum = df_sorted['max_frequency'].cumsum()
total_contributions = cumsum.iloc[-1]
cum_pct = cumsum / total_contributions * 100

n_50 = (cum_pct <= 50).sum() + 1
n_90 = (cum_pct <= 90).sum() + 1
n_total = len(df_sorted)

print(f"Max-contribution concentration:")
print(f"  {n_50} / {n_total} positions ({n_50/n_total:.1%}) account for 50% of max contributions")
print(f"  {n_90} / {n_total} positions ({n_90/n_total:.1%}) account for 90% of max contributions")

# Plot cumulative distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, n_total + 1), cum_pct.values, 'b-', linewidth=2)
ax.axhline(50, color='orange', linestyle='--', alpha=0.7, label=f'50% ({n_50} positions)')
ax.axhline(90, color='red', linestyle='--', alpha=0.7, label=f'90% ({n_90} positions)')
ax.axvline(n_50, color='orange', linestyle=':', alpha=0.5)
ax.axvline(n_90, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Positions (ranked by frequency)', fontsize=12)
ax.set_ylabel('Cumulative % of Max Contributions', fontsize=12)
ax.set_title('Concentration of Max-Pool Signal', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SWEEP_DIR / "max_contribution_cumulative.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: max_contribution_cumulative.png")

## Section 8: Save Results & Summary

In [ ]:
# Final summary
print("=" * 80)
print("CONSERVATION THRESHOLD SWEEP -- FINAL SUMMARY")
print("=" * 80)

print(f"\nTotal experiments: {len(df_results)}")
print(f"Thresholds tested: {THRESHOLDS}")
print(f"Pooling methods: {POOLING_METHODS}")
print(f"Models: XGBoost, MLP")
print(f"Seeds: {N_SPLITS}")

print("\n--- Overall Best Configurations ---")
for metric in ['roc_auc_mean', 'pr_auc_mean']:
    best_idx = df_summary[metric].idxmax()
    best = df_summary.loc[best_idx]
    metric_label = metric.replace('_mean', '').upper().replace('_', '-')
    print(f"\n  Best {metric_label} overall:")
    print(f"    threshold={best['threshold']:.2f}, pooling={best['pooling']}, model={best['model']}")
    print(f"    {metric_label} = {best[metric]:.4f} +/- {best[metric.replace('mean', 'std')]:.4f}")

print("\n--- Max-Contributing Residue Analysis ---")
print(f"  Best threshold for max-pool: {best_thresh}")
print(f"  Enzymes analyzed: {n_enzymes_analyzed}")
print(f"  Unique positions with max contributions: {len(max_position_counts)}")
print(f"  Positions for 50% of signal: {n_50}")
print(f"  Positions for 90% of signal: {n_90}")

print("\n--- Output Files ---")
output_files = [
    "residue_counts_per_threshold.csv",
    "residue_positions_per_threshold.csv",
    "threshold_sweep_results.csv",
    "threshold_sweep_summary.csv",
    "max_residue_importance.csv",
    "residue_count_vs_threshold.png",
    "threshold_sweep_lines.png",
    "threshold_sweep_gap.png",
    "pooling_advantage_vs_n_residues.png",
    "max_contributing_residues.png",
    "max_contribution_cumulative.png",
]
for f in output_files:
    path = SWEEP_DIR / f
    status = 'OK' if path.exists() else 'MISSING'
    print(f"  [{status}] {f}")

print("\nDone!")